In [27]:
# Imports
import sys
from pathlib import Path
BASE = Path.cwd().parent.parent
sys.path.insert(0, str(BASE)) # Go to base directory

# Numpy Pandas
import numpy as np
import pandas as pd

# Helper libraries
import re
import os
import nltk
import clip
import string
import joblib
import requests
import chromadb
from tqdm import tqdm
from PIL import Image
from io import BytesIO
from itertools import chain
from collections import Counter
from nltk.corpus import stopwords
from clip.simple_tokenizer import SimpleTokenizer

# Torch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision.models import resnet18, ResNet18_Weights

# Sklearn
from sklearn.feature_extraction.text import TfidfVectorizer

# Self imports
from src.data import Data


In [2]:
# Set device to "Cuda" if available.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [3]:
FINAL_DATA_PATH = "../../data"
d_train = joblib.load(f"{FINAL_DATA_PATH}/train_data_with_all_embeddings.pkl")
d_val = joblib.load(f"{FINAL_DATA_PATH}/val_data_with_all_embeddings.pkl")

In [13]:
## Write MLP module to load TFIDF 10k model

class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dims, output_dims, activation):
        super().__init__()
        layers = []

        prev_dim = input_dim
        for dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, dim))
            layers.append(activation())
            prev_dim = dim

        layers.append(nn.Linear(prev_dim, output_dims))

        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

In [20]:
# A Instance of Dataset since now we have 3 things to pass instead of 2
class QueryProductDataset(Dataset):
    def __init__(self, queries, products, labels):
        self.queries = queries
        self.products = products
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        query = torch.tensor(self.queries[idx].squeeze(), dtype=torch.float32)
        product = torch.tensor(self.products[idx].squeeze(), dtype=torch.float32)
        label = torch.tensor(self.labels[idx], dtype=torch.float32)

        return query, product, label

In [6]:
# Function to train the model and run validation per epoch
def train_model(model, train_data_loader, val_data_loader, optimizer, loss_criterion, epochs=10, device=device):
    train_losses = []
    val_losses = []

    for epoch in range(epochs):
        model.train()
        running_train_loss = 0
        for query_x, prod_x, labels in train_data_loader:
            query_x = query_x.to(device)
            prod_x = prod_x.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            query_emb = model(query_x)
            prod_emb = model(prod_x)

            loss = loss_criterion(query_emb, prod_emb, labels)
            loss.backward()

            optimizer.step()

            running_train_loss += (loss.item() * query_x.size(0))
        epoch_train_loss = running_train_loss / len(train_data_loader.dataset)
        train_losses.append(epoch_train_loss)

        model.eval()
        running_val_loss = 0
        with torch.no_grad():
            for query_x, prod_x, labels in val_data_loader:
                query_x = query_x.to(device)
                prod_x = prod_x.to(device)
                labels = labels.to(device)
                
                query_emb = model(query_x)
                prod_emb = model(prod_x)

                loss = loss_criterion(query_emb, prod_emb, labels)

                running_val_loss += (loss.item() * query_x.size(0))
        epoch_val_loss = running_val_loss / len(val_data_loader.dataset)
        val_losses.append(epoch_val_loss)

        print(f"Epoch [{epoch+1}/{epochs}]  "
            f"Train Loss: {epoch_train_loss:.4f}  "
            f"Val Loss: {epoch_val_loss:.4f}")

    return train_losses, val_losses

In [8]:
def stack_emb(col):
    dim = col.dropna().iloc[0].shape[0]
    return np.stack([np.zeros(dim) if x is None or isinstance(x, float) else x for x in col])

In [10]:
prod_train_comb_emb  = np.concatenate([stack_emb(d_train['Product_TFIDF_Emb']), stack_emb(d_train['Product_CLIP_text_Emb']), 
                                        stack_emb(d_train['Product_CLIP_img_Emb']), stack_emb(d_train['Product_Resnet18_Emb'])], axis=1)
query_train_comb_emb  = np.concatenate([stack_emb(d_train['Query_TFIDF_Emb']), stack_emb(d_train['Query_CLIP_text_Emb']), 
                                        stack_emb(d_train['Query_CLIP_img_Emb']), stack_emb(d_train['Query_Resnet18_Emb'])], axis=1)

prod_val_comb_emb  = np.concatenate([stack_emb(d_val['Product_TFIDF_Emb']), stack_emb(d_val['Product_CLIP_text_Emb']), 
                                        stack_emb(d_val['Product_CLIP_img_Emb']), stack_emb(d_val['Product_Resnet18_Emb'])], axis=1)
query_val_comb_emb  = np.concatenate([stack_emb(d_val['Query_TFIDF_Emb']), stack_emb(d_val['Query_CLIP_text_Emb']), 
                                        stack_emb(d_val['Query_CLIP_img_Emb']), stack_emb(d_val['Query_Resnet18_Emb'])], axis=1)

In [23]:
# ESCI map, determines how similar a product and query would be given there label
label_map = {"E":  1.0, "S":  1.0, "C":  1.0, "I": -1.0}
train_esci_labels = d_train["esci_label"].map(label_map).to_numpy()
val_esci_labels = d_val["esci_label"].map(label_map).to_numpy()

# Make the dataset
train_dataset = QueryProductDataset(query_train_comb_emb, prod_train_comb_emb, train_esci_labels)
val_dataset = QueryProductDataset(query_val_comb_emb, prod_val_comb_emb, val_esci_labels)

In [24]:
BATCH_SIZE = 128

# Create data loader
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=True)

In [25]:
INPUT_DIM =  512 + 768 + 768 + 512 # list(tfidf_model.modules())[-1].out_features + 2 * list(clip_model.modules())[-1].normalized_shape[0] + list(resnet_model.modules())[-3].num_features
OUTPUT_DIM = 512

## Choose the best MLP settings as per the above abalation
hidden_dim = [1024]
activation_fn = nn.ReLU

final_model = MLP(INPUT_DIM, hidden_dim, OUTPUT_DIM, activation_fn).to(device)
loss = nn.CosineEmbeddingLoss(0.2)
optimizer = optim.Adam(final_model.parameters(), lr=1e-4)

In [26]:
## Train that model
train_losses, val_losses = train_model(final_model, train_loader, val_loader, optimizer, loss, epochs=3)

Epoch [1/3]  Train Loss: 0.2210  Val Loss: 0.3174
Epoch [2/3]  Train Loss: 0.1521  Val Loss: 0.2899
Epoch [3/3]  Train Loss: 0.1203  Val Loss: 0.2607


In [28]:
## Save that model
FINAL_MODEL_PATH = "../../model"
os.makedirs(FINAL_MODEL_PATH, exist_ok=True)

torch.save(final_model.state_dict(), f'{FINAL_MODEL_PATH}/final_model.pt')